In [11]:
def tile_start(length, tile  = 512, stride = 384):
    
    if length <=tile:
        return [0]
    
    
    starts = []
    
    start = 0
    
    while start + tile  <= length:
        
        starts.append(start)
        
        start = start + stride
        
    if starts[-1] + tile < length:
                    
        starts.append(length - tile)
                    
           
    
    return starts
    
        
     
   
        
        
    
     
        
         

In [12]:
def tile_corner(width, height):

    corners = []
   
    x_starts = tile_start(width)
    y_starts = tile_start(height)
    
    for y in y_starts:
        for x in x_starts:
            corners.append((x, y))
        
    
    
    
    return corners

In [13]:
tile_start(1000), tile_start(896), tile_start(1300), tile_start(512), tile_start(278)

([0, 384, 488], [0, 384], [0, 384, 768, 788], [0], [0])

In [14]:
print(tile_corner(1000, 896))   
print(tile_corner(278, 300))

[(0, 0), (384, 0), (488, 0), (0, 384), (384, 384), (488, 384)]
[(0, 0)]


In [15]:
%pip install -q -U huggingface_hub

#Kaggle blocks this AWS server's IP (every kaggle.com request gets a 403, even the homepage),
#so kagglehub can't download from here no matter what token is used.
#Instead grab the same iSAID data from Hugging Face:
#   - iSAID annotations (COCO json + instance masks) from isaaccorley/isaid
#   - iSAID images are the DOTA v1.0 images, from isaaccorley/dota
import os
import tarfile
from huggingface_hub import hf_hub_download

path = os.path.expanduser("~/data/isaid")
os.makedirs(path, exist_ok=True)

files = [
    ("isaaccorley/isaid", "isaid_annotations_train.tar.gz"),
    ("isaaccorley/isaid", "isaid_annotations_val.tar.gz"),
    ("isaaccorley/dota", "dotav1.0_images_train.tar.gz"),   #~10 GB
    ("isaaccorley/dota", "dotav1.0_images_val.tar.gz"),     #~3.3 GB
]

for repo, name in files:
    done = os.path.join(path, name + ".done") #marker so re-running the cell skips what's already extracted
    if os.path.exists(done):
        continue
    archive = hf_hub_download(repo_id=repo, filename=name, repo_type="dataset")
    with tarfile.open(archive) as tar:
        tar.extractall(path, filter="data")
    open(done, "w").close()

#Layout: {path}/train|val/images/*.png, {path}/train|val/Annotations/iSAID_*.json, {path}/train|val/Instance_masks/
print("Path to dataset files:", path)


Note: you may need to restart the kernel to use updated packages.
Path to dataset files: /home/ubuntu/data/isaid


In [16]:
import os 

train_path = os.path.join(path, "train")

os.listdir(train_path)








['images', 'Annotations', 'Instance_masks', 'Semantic_masks']

In [17]:
train_image_path= os.path.join(train_path, "images")

len(os.listdir(train_image_path))


1411

In [18]:
annotation_path = os.path.join(train_path, "Annotations")

os.listdir(annotation_path)


['iSAID_train.json']

In [19]:
import json 

json_path = os.path.join(annotation_path, 'iSAID_train.json')

with open(json_path) as f:
    coco = json.load(f)

In [ ]:
import random


# Getting the ids for my images

def get_img_id():
    
    image_ids = []
    
    for image in coco["images"]:
    
        image_ids.append(image["id"])
    
    return image_ids   

  

image_ids = get_img_id()

# Shuffling the images to prevent overfitting 

random.Random(42).shuffle(image_ids)  

# Spitting my ids into my train and holdou
# This should be done before tiling because I don't want tiles of the same image to be in different sets 

holdout_length = int(len(image_ids) / 10)

holdout_ids = image_ids[:holdout_length]

train_ids = image_ids[holdout_length:]

# Verifying the length of the Ids

print(f"Train_ids: {len(train_ids)}") 
      
print(f"Hold_out_ids: {len(holdout_ids)}")

print(f"image_ids: {len(image_ids)}")


# Checking the split worked
# The & will tell me if any of the sets have any ids in common, if so then splitting the images didn't work 

print(set(train_ids) & set(holdout_ids))



     

    
    
    
    



    








Train_ids: 1270
Hold_out_ids: 141
image_ids: 1411
set()
